In [2]:
!pip install xgboost

In [3]:
# 必要なライブラリをインポート
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [4]:
# 1. クラスタリング結果を読み込み

cluster_all_df = pd.read_csv('prefecture_cluster_all_methods.csv', encoding='utf-8-sig')
print("\n✓ クラスタリング結果を読み込みました")
print(f"  利用可能なクラスタリング方法: {len(cluster_all_df.columns)-1}種類")

available_methods = [col for col in cluster_all_df.columns if col != '都道府県名']
for i, method in enumerate(available_methods, 1):
    print(f"  {i}. {method}")


✓ クラスタリング結果を読み込みました
  利用可能なクラスタリング方法: 4種類
  1. クラスタ_簡単版
  2. クラスタ_件数考慮
  3. クラスタ_階層
  4. クラスタ_全特徴


In [5]:
df.columns

Index(['市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引価格（総額）_log', '取引時点_年', '建築年_西暦', '築年数_欠損', '取引時点での築年数',
       '取引の事情等_調停・競売等', '取引の事情等_関係者間取引', '取引の事情等_その他', '改装_改装済', '改装_未改装',
       '間取り_grouped_その他', '間取り_grouped_オープンフロア', '間取り_grouped_欠損値',
       '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ', '間取り_grouped_１ＬＤＫ',
       '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ', '間取り_grouped_２Ｋ',
       '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ', '間取り_grouped_３ＤＫ',
       '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ', '間取り_grouped_４ＬＤＫ', '都市計画_高価格帯',
       '都市計画_中価格帯', '都市計画_低価格帯', '人口密度', '市区町村人口密度', '犯罪発生率', '築年数_2乗',
       '築年数_3乗', '築年数_log', '面積_log', '面積_平方根', '築年数×面積', '築年数×駅距離',
       '築年数×建ぺい率', '築年数×容積率', '築年数×人口密度', '面積×駅距離', '面積×建ぺい率', '面積×容積率',
       '面積×人口密度', '建築可能性', '容積率_建ぺい率比', '建ぺい率_2乗', '容積率_2乗', '駅距離_逆数',
       '駅距離_log', '駅距離_2乗', '駅距離×建ぺい率', '駅距離×容積率', '人口密度_log', '市区町村人口密度_l

In [6]:

# --- 説明変数・目的変数のセット ---
drop_cols = ['市区町村コード', 
             '間取り','用途', '今後の利用目的', 
             '都市計画','取引時点', '取引価格（総額）_log', '取引の事情等_その他',
             '改装_未改装', '間取り_grouped_その他','建築年'
             ]

X_cols = [col for col in df.columns if col not in drop_cols]
print(f"\n使用する特徴量の数: {len(X_cols)}個")

X = df[X_cols].copy()
Y = df['取引価格（総額）_log'].copy()


使用する特徴量の数: 72個


In [7]:
# カテゴリカラムを 'category' 型に変換
categorical_cols = ['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '建物の構造']

for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

# 数値カラムのみクリーニング
numerical_cols = X.select_dtypes(include=[np.number]).columns
X[numerical_cols] = X[numerical_cols].replace([np.inf, -np.inf], np.nan)
X[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())

In [8]:
##ベースラインモデル（クラスタリングなし）
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

model_base = XGBRegressor(
    enable_categorical=True,
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    missing=np.nan
)

model_base.fit(
    X_train_base, y_train_base,
    eval_set=[(X_test_base, y_test_base)],
    verbose=False 
)

y_pred_base = model_base.predict(X_test_base)
mae_base = mean_absolute_error(y_test_base, y_pred_base)

print(f"\n【ベースライン結果】")
print(f"MAE: {mae_base:.6f}")


【ベースライン結果】
MAE: 0.076826


In [9]:

# #クラスタリング方法選択バージョン

# ★★★ クラスタリング方法を選択 ★★★
SELECTED_METHOD = 'クラスタ_件数考慮'  # ← ここでどのクラスタリング方法を適用するか指定できる

# 選択肢: 'クラスタ_簡単版', 'クラスタ_件数考慮', 'クラスタ_階層', 'クラスタ_全特徴'

print(f"\n選択されたクラスタリング方法: {SELECTED_METHOD}")

# クラスタ情報を結合
cluster_mapping = cluster_all_df.set_index('都道府県名')[SELECTED_METHOD].to_dict()
df['クラスタ'] = df['都道府県名'].map(cluster_mapping)

print(f"\nクラスタの分布:")
for cluster_id in sorted(df['クラスタ'].unique()):
    count = (df['クラスタ'] == cluster_id).sum()
    print(f"  クラスタ {cluster_id}: {count:,}件")



選択されたクラスタリング方法: クラスタ_件数考慮

クラスタの分布:
  クラスタ 0: 170,446件
  クラスタ 1: 71,376件
  クラスタ 2: 324,858件


In [10]:
!pip install tensorflow keras

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, Concatenate, Embedding, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error

## クラスタごとにモデル構築 (メモリ効率化: クラスタ2: XGBoost, その他: NN + Embedding)

print(f"【{SELECTED_METHOD}】でクラスタごとにモデルを構築（クラスタ2: XGBoost, その他: NN + Embedding）")

all_predictions = []
all_actuals = []
cluster_results = {}
scalers = {}
label_encoders = {}
NN_CATEGORICAL_COLS = ['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '建物の構造']


【クラスタ_件数考慮】でクラスタごとにモデルを構築（クラスタ2: XGBoost, その他: NN + Embedding）


In [17]:
# NNの埋め込みモデル構築関数 (レイヤー名を英語に修正)
def create_embedding_model(numerical_input_dim, cat_feature_info):
    
    # 1. 数値特徴量の入力
    numerical_input = Input(shape=(numerical_input_dim,), name='numerical_input')
    
    # 2. カテゴリ特徴量の入力と埋め込み層
    embedding_layers = []
    category_inputs = []
    
    # カテゴリ列のインデックス用カウンター
    cat_idx = 0 
    
    for col_japanese, vocab_size, embedding_dim in cat_feature_info:
        # ----------------------------------------------------
        # 💡 修正点: 日本語の列名を安全な英語のプレフィックスに変換
        # ----------------------------------------------------
        # 例: '都道府県名' -> 'cat_0'
        col_safe_name = f'cat_{cat_idx}' 
        cat_idx += 1 

        input_layer = Input(shape=(1,), name=f'{col_safe_name}_input')
        category_inputs.append(input_layer)
        
        # 埋め込みベクトルの次元 (前回からのロジックを維持)
        if vocab_size <= 10:
            e_dim = 2
        else:
            e_dim = min(50, int(vocab_size**0.25) * 2) 
        
        # レイヤー名も安全な名前に変更
        embedding = Embedding(
            input_dim=vocab_size,
            output_dim=e_dim,     
            input_length=1,
            name=f'{col_safe_name}_embedding' # 修正された安全な名前
        )(input_layer)
        
        embedding = Flatten()(embedding)
        embedding_layers.append(embedding)

    # 3. 全特徴量を結合
    if embedding_layers:
        all_features = Concatenate()([numerical_input] + embedding_layers)
    else:
        all_features = numerical_input

    # 4. 共通の深層ネットワーク
    dense = Dense(128, activation='relu')(all_features)
    dense = Dropout(0.2)(dense)
    dense = Dense(64, activation='relu')(dense)
    dense = Dropout(0.2)(dense)
    output = Dense(1, activation='linear')(dense)

    # モデル定義: 入力リストと出力
    model = Model(inputs=[numerical_input] + category_inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mae', metrics=['mae'])
    return model

In [18]:

# NN用の整数エンコーディングをデータフレーム全体ではなく、必要な列に直接適用
# X.copy() を避け、メモリ効率を高める
X_nn_temp = X.copy(deep=False) # 浅いコピーでメモリ負荷を減らす（ただし安全ではないため次のステップで修正）

# ----------------------------------------------------
# ⚠️ メモリ効率化のため、Xに直接整数エンコーディングの結果を格納
#    Xに新たな列を追加します。
# ----------------------------------------------------
for col in NN_CATEGORICAL_COLS:
    le = LabelEncoder()
    # 欠損値対策
    X[col] = X[col].astype('object').fillna('Missing')
    # LabelEncoderはデータセット全体でfit
    le.fit(X[col].unique())
    X[f'{col}_int'] = le.transform(X[col])
    label_encoders[col] = le
    
# 数値特徴量と整数エンコードされたカテゴリ特徴量のリストを定義
NN_NUMERICAL_COLS = [col for col in X.columns 
                     if col not in NN_CATEGORICAL_COLS and col not in [f'{c}_int' for c in NN_CATEGORICAL_COLS]
                     and X[col].dtype in ['float64', 'int64', 'int32', 'float32']] # 数値型の列を明示的に選択
NN_INTEGER_COLS = [f'{c}_int' for c in NN_CATEGORICAL_COLS]



In [33]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Input, Concatenate, Embedding, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
import numpy as np
import joblib
import os # モデル保存に必要

## クラスタごとにモデル構築 (修正版: 訓練・テストMAEを取得)

# ⚠️ 注意: df, X, Y, SELECTED_METHOD が前もって定義されている必要があります
print(f"【{SELECTED_METHOD}】でクラスタごとにモデルを構築（クラスタ2: XGBoost, その他: NN + Embedding）")

all_predictions = []
all_actuals = []
cluster_results = {}
scalers = {}
label_encoders = {}
NN_CATEGORICAL_COLS = ['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '建物の構造']

# NNの埋め込みモデル構築関数 (再掲 - 変更なし)
def create_embedding_model(numerical_input_dim, cat_feature_info):
    numerical_input = Input(shape=(numerical_input_dim,), name='numerical_input')
    embedding_layers = []
    category_inputs = []
    cat_idx = 0 
    
    for col_japanese, vocab_size, embedding_dim in cat_feature_info:
        col_safe_name = f'cat_{cat_idx}' 
        cat_idx += 1 

        input_layer = Input(shape=(1,), name=f'{col_safe_name}_input')
        category_inputs.append(input_layer)
        
        if vocab_size <= 10:
            e_dim = 2
        else:
            e_dim = min(50, int(vocab_size**0.25) * 2) 
        
        embedding = Embedding(
            input_dim=vocab_size,
            output_dim=e_dim,     
            input_length=1,
            name=f'{col_safe_name}_embedding'
        )(input_layer)
        
        embedding = Flatten()(embedding)
        embedding_layers.append(embedding)

    if embedding_layers:
        all_features = Concatenate()([numerical_input] + embedding_layers)
    else:
        all_features = numerical_input

    dense = Dense(128, activation='relu')(all_features)
    dense = Dropout(0.2)(dense)
    dense = Dense(64, activation='relu')(dense)
    dense = Dropout(0.2)(dense)
    output = Dense(1, activation='linear')(dense)

    model = Model(inputs=[numerical_input] + category_inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mae', metrics=['mae'])
    return model


# NN用の整数エンコーディングをデータフレーム全体に適用 (再掲)
for col in NN_CATEGORICAL_COLS:
    le = LabelEncoder()
    # 欠損値対策: 'Missing'を挿入するために、一時的にobject型に変換
    X[col] = X[col].astype('object').fillna('Missing') 
    X[col] = X[col].astype(str)
    
    le.fit(X[col].unique())
    X[f'{col}_int'] = le.transform(X[col])
    label_encoders[col] = le
    
NN_NUMERICAL_COLS = [col for col in X.columns 
                     if col not in NN_CATEGORICAL_COLS and col not in [f'{c}_int' for c in NN_CATEGORICAL_COLS]
                     and X[col].dtype in ['float64', 'int64', 'int32', 'float32']] 
NN_INTEGER_COLS = [f'{c}_int' for c in NN_CATEGORICAL_COLS]


for cluster_id in sorted(df['クラスタ'].unique()):
    print(f"\n{'='*80}")
    print(f"【クラスタ {cluster_id}】")
    print('='*80)
    
    cluster_mask = df['クラスタ'] == cluster_id
    X_cluster = X[cluster_mask].copy()
    Y_cluster = Y[cluster_mask].copy()

    # 訓練/テスト分割
    X_train, X_test, y_train, y_test = train_test_split(
        X_cluster, Y_cluster, test_size=0.2, random_state=42
    )

    train_mae = None
    test_mae = None

    # クラスタ2以外 (0と1) はニューラルネットワーク (NN) を使用
    if cluster_id != 2:
        
        # データのスケーリング (数値特徴量のみ)
        scaler = StandardScaler()
        X_train_num_scaled = scaler.fit_transform(X_train[NN_NUMERICAL_COLS].values)
        X_test_num_scaled = scaler.transform(X_test[NN_NUMERICAL_COLS].values)
        scalers[cluster_id] = scaler
        
        # 埋め込み層のパラメータ設定
        cat_feature_info = []
        for col in NN_CATEGORICAL_COLS:
            vocab_size = len(label_encoders[col].classes_)
            cat_feature_info.append((col, vocab_size, None)) 

        # モデル構築
        model = create_embedding_model(X_train_num_scaled.shape[1], cat_feature_info)
        
        print(f"モデルタイプ: ニューラルネットワーク (Keras + Embedding)")
        print(f"データ件数: {len(X_cluster):,}件 (訓練: {len(X_train):,} / テスト: {len(X_test):,})")
        
        # Kerasへの入力形式に整形
        train_inputs = [X_train_num_scaled] + [X_train[f'{col}_int'].values.reshape(-1, 1) for col in NN_CATEGORICAL_COLS]
        test_inputs = [X_test_num_scaled] + [X_test[f'{col}_int'].values.reshape(-1, 1) for col in NN_CATEGORICAL_COLS]

        print("\nモデル学習中...")
        
        # NNの学習と履歴の取得 
        history = model.fit(
            train_inputs, y_train,
            epochs=50, 
            batch_size=32,
            validation_data=(test_inputs, y_test),
            verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
        )
        
        # Kerasの訓練・テストMAEの取得 (EarlyStoppingが効いた最終エポックの値を取得)
        train_mae = history.history['mae'][-1]
        test_mae = history.history['val_mae'][-1]
        
        # 予測
        y_pred = model.predict(test_inputs, verbose=0).flatten()
        
    # クラスタ2 のみ XGBoost を使用
    else: # <--- 修正された正しいインデントレベル
        # XGBoost用のデータセットを準備
        NN_INTEGER_COLS = [f'{c}_int' for c in NN_CATEGORICAL_COLS]
        X_cluster = X_cluster.drop(columns=NN_INTEGER_COLS, errors='ignore')
        Y_cluster = Y_cluster.copy()

        # XGBoost用にカテゴリ列を明示的に 'category' 型に変換
        XGB_CATEGORICAL_COLS = NN_CATEGORICAL_COLS
        for col in XGB_CATEGORICAL_COLS:
            X_cluster[col] = X_cluster[col].astype('category')
        
        # 訓練/テスト分割 (再実施、XGBoostに合わせたX_clusterを使用)
        X_train, X_test, y_train, y_test = train_test_split(
            X_cluster, Y_cluster, test_size=0.2, random_state=42
        )
        
        # モデル構築 
        model = XGBRegressor(
            enable_categorical=True, 
            objective='reg:squarederror',
            eval_metric='mae',  
            n_estimators=1000,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            early_stopping_rounds=50,
            missing=np.nan
        )
        
        print(f"モデルタイプ: XGBoost")
        print(f"データ件数: {len(X_cluster):,}件 (訓練: {len(X_train):,} / テスト: {len(X_test):,})")
        
        print("\nモデル学習中...")
        
        eval_set_list = [(X_train, y_train), (X_test, y_test)]
        
        # XGBoostの学習と評価結果の取得
        model.fit(
            X_train, y_train,
            eval_set=eval_set_list,
            verbose=False,
        )
        
        # model.evals_result() を使って結果を取得
        evals_result = model.evals_result()
        
        # XGBoostの訓練・テストMAEの取得 (EarlyStoppingが効いた最終エポックの値を取得)
        best_iteration = model.best_iteration
        train_mae = evals_result['validation_0']['mae'][best_iteration]
        test_mae = evals_result['validation_1']['mae'][best_iteration]
        
        # 予測
        y_pred = model.predict(X_test)
        
    # 共通の評価・結果保存 
    print(f"\n【結果】")
    print(f"訓練MAE: {train_mae:.6f} / テストMAE: {test_mae:.6f}")
    
    all_predictions.extend(y_pred)
    all_actuals.extend(y_test)
    cluster_results[cluster_id] = {
        'train_mae': train_mae, 
        'test_mae': test_mae,   
        'test_size': len(X_test),
        'train_size': len(X_train)
    }
    
    # モデル保存ロジック (joblibやosモジュールが必要)
    MODEL_DIR = '../models/'
    os.makedirs(MODEL_DIR, exist_ok=True)

    if cluster_id != 2:
        model_name = f'model_cluster_{cluster_id}_{SELECTED_METHOD}.h5'
        model_path = os.path.join(MODEL_DIR, model_name)
        model.save(model_path)
        
        scaler_name = f'scaler_cluster_{cluster_id}_{SELECTED_METHOD}.joblib'
        scaler_path = os.path.join(MODEL_DIR, scaler_name)
        joblib.dump(scaler, scaler_path)
        
        for col, le in label_encoders.items():
            le_name = f'le_{col}.joblib'
            le_path = os.path.join(MODEL_DIR, le_name)
            joblib.dump(le, le_path)
            
        print(f"✅ NNモデルとスケーラー、ラベルエンコーダーを保存しました。")
        
    else:
        model_name = f'model_cluster_{cluster_id}_{SELECTED_METHOD}.joblib'
        model_path = os.path.join(MODEL_DIR, model_name)
        joblib.dump(model, model_path)
        print(f"✅ XGBoostモデルを {model_name} として保存しました。")

【クラスタ_件数考慮】でクラスタごとにモデルを構築（クラスタ2: XGBoost, その他: NN + Embedding）

【クラスタ 0】
モデルタイプ: ニューラルネットワーク (Keras + Embedding)
データ件数: 170,446件 (訓練: 136,356 / テスト: 34,090)

モデル学習中...



【結果】
訓練MAE: 0.077777 / テストMAE: 0.084461
✅ NNモデルとスケーラー、ラベルエンコーダーを保存しました。

【クラスタ 1】
モデルタイプ: ニューラルネットワーク (Keras + Embedding)
データ件数: 71,376件 (訓練: 57,100 / テスト: 14,276)

モデル学習中...



【結果】
訓練MAE: 0.086075 / テストMAE: 0.092419
✅ NNモデルとスケーラー、ラベルエンコーダーを保存しました。

【クラスタ 2】
モデルタイプ: XGBoost
データ件数: 324,858件 (訓練: 259,886 / テスト: 64,972)

モデル学習中...

【結果】
訓練MAE: 0.058251 / テストMAE: 0.068963
✅ XGBoostモデルを model_cluster_2_クラスタ_件数考慮.joblib として保存しました。


In [37]:
!pip install tabulate

In [38]:
# --- クラスタ別モデル評価結果の整形表示 (訓練・テスト) ---

from sklearn.metrics import mean_absolute_error
import pandas as pd

# クラスタ別MAEの表示
print("\n" + "="*70)
print(f"クラスタ別モデル性能評価 (クラスタ2: XGBoost, その他: NN + Embedding)")
print("="*70)

# DataFrameを作成して整形
results_df = pd.DataFrame.from_dict(cluster_results, orient='index')
results_df.index.name = 'クラスタID'

# 列名の整理とモデルタイプの追加
results_df['モデル'] = results_df.index.map(lambda x: 'XGBoost' if x == 2 else 'NN (Embedding)')
results_df = results_df.rename(columns={
    'train_mae': '訓練MAE', 
    'test_mae': 'テストMAE', 
    'train_size': '訓練件数',
    'test_size': 'テスト件数'
})

# 件数のフォーマット
for col in ['訓練件数', 'テスト件数']:
    results_df[col] = results_df[col].apply(lambda x: f'{int(x):,}')

# 全体MAEの計算 (全テストデータに対する加重平均MAE)
# all_actualsとall_predictionsがリストとして存在することを前提
overall_mae = mean_absolute_error(all_actuals, all_predictions)

# 表示
output_cols = ['モデル', '訓練件数', '訓練MAE', 'テスト件数', 'テストMAE']
print(results_df[output_cols].to_markdown(floatfmt=".6f"))
print("\n" + "-"*70)
print(f"全体 (加重平均) テストMAE: {overall_mae:.6f}")
print("-"*70)


クラスタ別モデル性能評価 (クラスタ2: XGBoost, その他: NN + Embedding)
|   クラスタID | モデル         | 訓練件数   |   訓練MAE | テスト件数   |   テストMAE |
|-------------:|:---------------|:-----------|----------:|:-------------|------------:|
|            0 | NN (Embedding) | 136,356    |  0.077777 | 34,090       |    0.084461 |
|            1 | NN (Embedding) | 57,100     |  0.086075 | 14,276       |    0.092419 |
|            2 | XGBoost        | 259,886    |  0.058251 | 64,972       |    0.068963 |

----------------------------------------------------------------------
全体 (加重平均) テストMAE: 0.075305
----------------------------------------------------------------------
